# Ablation IMV - MAGIC Gamma / tabular feature groups

Feature-group ablation on the public
[UCI MAGIC Gamma Telescope dataset](https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope)
([DOI 10.24432/C52C8B](https://doi.org/10.24432/C52C8B)), scored with the
installed `imv` package. The dataset is CC BY 4.0 licensed and contains 19,020
simulated telescope events, ten continuous image measurements, no missing
values, and a binary gamma-versus-hadron target.

This example keeps one MLP architecture fixed and removes one physically
related predictor group at a time. That isolates the information contributed
by geometry, concentration, moments, and orientation without mixing feature
removal with model-family changes.

**Pipeline**: download the checksum-validated UCI data -> make leakage-free
stratified splits -> train the full and four ablated models over ten seeds ->
calculate and visualise directional ablation IMV.

This is an example run, not a replication of a published result. The UCI
dataset description cautions that the simulated class balance is not
representative of operation in the field, so ROC AUC and balanced accuracy are
reported only as familiar diagnostics; IMV itself is calculated from the
held-out probabilistic predictions. Downloaded data, restartable predictions,
and exported artifacts are cached outside the repository under `~/.cache/imv`.


In [ ]:
import gc
import hashlib
import os
import shutil
import sys
import tempfile
import time
import warnings
from pathlib import Path
from types import SimpleNamespace
from urllib.request import urlretrieve
from zipfile import ZipFile


_PATH_DISPLAY_ROOT = Path()


def relative_path(path, *, start=None):
    '''Return a display-only path relative to an explicit local anchor.'''

    anchor = Path(start) if start is not None else _PATH_DISPLAY_ROOT
    try:
        relative = os.path.relpath(
            Path(path).expanduser().resolve(), start=anchor.expanduser().resolve()
        )
    except ValueError:
        relative = Path(path).name
    return Path(relative).as_posix()


def _relative_warning_text(message):
    text = str(message)
    roots = {
        Path.home(),
        Path(sys.prefix),
        Path(sys.base_prefix),
        _PATH_DISPLAY_ROOT,
        Path(tempfile.gettempdir()),
    }
    for root in sorted(roots, key=lambda value: len(str(value)), reverse=True):
        absolute = str(root.expanduser().resolve())
        text = text.replace(absolute, relative_path(absolute))
    return text


def _show_relative_warning(message, category, filename, lineno, file=None, line=None):
    stream = file if file is not None else sys.stderr
    warning_text = _relative_warning_text(message)
    print(
        f"{relative_path(filename)}:{lineno}: {category.__name__}: {warning_text}",
        file=stream,
    )


warnings.showwarning = _show_relative_warning


%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

import imv
from imv import AblationIMV, information_deficit, ll
from imv.utils import plot_ablation_matrix, save_figure


# Fail fast if Jupyter is attached to a different installation of imv.
_HERE = Path().resolve()
REPOSITORY_ROOT = next(
    (
        path
        for path in (_HERE, *_HERE.parents)
        if (path / "pyproject.toml").is_file() and (path / "src" / "imv").is_dir()
    ),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Run this notebook from inside the imv repository checkout.")
_PATH_DISPLAY_ROOT = REPOSITORY_ROOT
IMV_SOURCE = Path(imv.__file__).resolve().parent
EXPECTED_IMV_SOURCE = (REPOSITORY_ROOT / "src" / "imv").resolve()
if IMV_SOURCE != EXPECTED_IMV_SOURCE:
    raise RuntimeError(
        "Expected repository package at "
        f"{relative_path(EXPECTED_IMV_SOURCE)}, imported {relative_path(IMV_SOURCE)}"
    )


CACHE = Path(os.environ.get("IMV_CACHE_HOME", Path.home() / ".cache" / "imv"))
DATA_HOME = Path(os.environ.get("IMV_DATA_CACHE", CACHE / "datasets")) / "uci_magic_gamma"
ARTIFACTS = Path(
    os.environ.get("IMV_ARTIFACT_CACHE", CACHE / "notebook_artifacts")
) / "ablation_imv_magic_gamma"
RESULTS = ARTIFACTS / "results"
RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES = ARTIFACTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
PREDICTIONS = RESULTS / "feature_groups_mlp64x32_e12_v1"
PREDICTIONS.mkdir(parents=True, exist_ok=True)


SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
BATCH, TEST_BATCH, EPOCHS, LR = 256, 1024, 12, 1e-3
torch.set_num_threads(min(4, os.cpu_count() or 1))


ablator = AblationIMV(random_seed=SEEDS[0])
print(f"imv {imv.__version__} from {relative_path(IMV_SOURCE)}")
print("package class:", f"{AblationIMV.__module__}.{AblationIMV.__name__}")


## 1. Download and validate

The notebook downloads the 646 KB archive directly from UCI and validates the
SHA-256 digest of `magic04.data` before parsing it. No data is committed to the
repository. The ten predictor names and group definitions follow the UCI data
dictionary.


In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/159/magic+gamma+telescope.zip"
DATA_SHA256 = "e9314b7ebd4b4b59a3b3d65f7316663963777b16a46786877651dbbaa640b36a"
DATA_MEMBER = "magic04.data"


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def ensure_magic_data():
    DATA_HOME.mkdir(parents=True, exist_ok=True)
    data_path = DATA_HOME / DATA_MEMBER
    if data_path.exists() and file_sha256(data_path) == DATA_SHA256:
        return data_path

    archive_path = DATA_HOME / "magic_gamma_telescope.zip.download"
    extracted_path = DATA_HOME / f"{DATA_MEMBER}.download"
    try:
        urlretrieve(DATA_URL, archive_path)
        with ZipFile(archive_path) as archive:
            if DATA_MEMBER not in archive.namelist():
                raise RuntimeError(f"UCI archive does not contain {DATA_MEMBER}")
            with archive.open(DATA_MEMBER) as source, extracted_path.open("wb") as target:
                shutil.copyfileobj(source, target)
        actual = file_sha256(extracted_path)
        if actual != DATA_SHA256:
            raise RuntimeError(
                f"Checksum mismatch for {DATA_MEMBER}: expected {DATA_SHA256}, got {actual}"
            )
        extracted_path.replace(data_path)
    finally:
        archive_path.unlink(missing_ok=True)
        extracted_path.unlink(missing_ok=True)
    return data_path


FEATURES = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
]
data_path = ensure_magic_data()
data = pd.read_csv(data_path, header=None, names=[*FEATURES, "class"])

if data.shape != (19020, 11):
    raise RuntimeError(f"Expected 19,020 rows and 11 columns, got {data.shape}")
if set(data["class"]) != {"g", "h"}:
    raise RuntimeError(f"Unexpected target labels: {sorted(data['class'].unique())}")
data[FEATURES] = data[FEATURES].apply(pd.to_numeric, errors="raise")
if data[FEATURES].isna().any().any():
    raise RuntimeError("Unexpected missing predictor values in MAGIC Gamma data")
data["target"] = (data.pop("class") == "g").astype("int64")

print("source:", relative_path(data_path))
print(data.shape, "class counts:", data["target"].value_counts().sort_index().to_dict())
data.head()


## 2. Define the feature-group ablations

The four groups partition all ten predictors. Every variant uses the same
train/test rows, train-fitted scaler, optimisation budget, and 64-by-32 MLP;
only the named input group is omitted.


In [ ]:
VARIANT_DROPS = {
    "FullFeatures": [],
    "NoGeometry": ["fLength", "fWidth"],
    "NoConcentration": ["fSize", "fConc", "fConc1"],
    "NoMoments": ["fAsym", "fM3Long", "fM3Trans"],
    "NoOrientation": ["fAlpha", "fDist"],
}
VARIANTS = list(VARIANT_DROPS)


def kept_features(variant):
    dropped = set(VARIANT_DROPS[variant])
    return [feature for feature in FEATURES if feature not in dropped]


class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(np.asarray(features), dtype=torch.float32)
        self.labels = torch.tensor(np.asarray(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {"features": self.features[index], "labels": self.labels[index]}


def make_split(seed):
    train_frame, test_frame = train_test_split(
        data,
        test_size=0.25,
        stratify=data["target"],
        random_state=seed,
    )
    scaler = StandardScaler()
    train_scaled = pd.DataFrame(
        scaler.fit_transform(train_frame[FEATURES]),
        columns=FEATURES,
        index=train_frame.index,
    )
    test_scaled = pd.DataFrame(
        scaler.transform(test_frame[FEATURES]),
        columns=FEATURES,
        index=test_frame.index,
    )
    return (
        train_scaled,
        test_scaled,
        train_frame["target"].to_numpy(),
        test_frame["target"].to_numpy(),
    )


class GammaMLP(nn.Module):
    '''One fixed architecture used with each retained predictor set.'''

    def __init__(self, n_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Linear(32, 2),
        )

    def forward(self, features, labels=None):
        logits = self.network(features)
        loss = None if labels is None else F.cross_entropy(logits, labels)
        return SimpleNamespace(loss=loss, logits=logits)


variant_rows = []
for variant in VARIANTS:
    retained = kept_features(variant)
    model = GammaMLP(len(retained))
    variant_rows.append(
        {
            "variant": variant,
            "omitted": ", ".join(VARIANT_DROPS[variant]) or "none",
            "retained_features": len(retained),
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
        }
    )
variant_table = pd.DataFrame(variant_rows).set_index("variant")
variant_table


## 3. Train and predict with the package

For each seed, all five prediction frames refer to exactly the same held-out
rows and labels. A fresh stratified split measures sensitivity to sampling as
well as optimisation. Standardisation is fitted on the training rows only.

Predictions are cached by run specification, and cached files are reused only
after their schema, probabilities, labels, and class decisions pass validation.


In [ ]:
def prediction_path(variant, seed):
    return PREDICTIONS / f"predictions_magic_gamma_{variant.lower()}_seed_{seed}.csv"


def valid_cached_predictions(path, expected_labels):
    if not path.exists():
        return None
    try:
        frame = pd.read_csv(path)
    except (OSError, ValueError, pd.errors.ParserError):
        return None
    required = [
        "Negative Probability",
        "Positive Probability",
        "True Label",
        "Predicted Label",
    ]
    if list(frame.columns) != required or len(frame) != len(expected_labels):
        return None
    positive = frame["Positive Probability"].to_numpy(float)
    negative = frame["Negative Probability"].to_numpy(float)
    labels = frame["True Label"].to_numpy(int)
    predicted = frame["Predicted Label"].to_numpy(int)
    probabilities = np.column_stack([negative, positive])
    if (
        not np.isfinite(probabilities).all()
        or np.any((probabilities < 0) | (probabilities > 1))
        or not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
        or not np.array_equal(labels, expected_labels)
        or not np.array_equal(predicted, probabilities.argmax(axis=1))
    ):
        return None
    return frame


previous_path = RESULTS / "magic_gamma_variant_diagnostics.csv"
if previous_path.exists():
    previous = pd.read_csv(previous_path)
    previous_minutes = {
        (int(row.seed), row.variant): float(row.minutes)
        for row in previous.itertuples()
        if pd.notna(row.minutes)
    }
else:
    previous_minutes = {}


seed_matrices, matrix_records, diagnostics = [], [], []
for seed in SEEDS:
    train_x, test_x, train_y, test_y = make_split(seed)
    frames = {}
    print(f"seed {seed}")
    for variant in VARIANTS:
        retained = kept_features(variant)
        path = prediction_path(variant, seed)
        started = time.perf_counter()
        cached = valid_cached_predictions(path, test_y)
        if cached is not None:
            frame = cached
        else:
            ablator.set_seed(seed)
            model = GammaMLP(len(retained))
            train_loader = DataLoader(
                TabularDataset(train_x[retained].to_numpy(), train_y),
                batch_size=BATCH,
                shuffle=True,
            )
            test_loader = DataLoader(
                TabularDataset(test_x[retained].to_numpy(), test_y),
                batch_size=TEST_BATCH,
                shuffle=False,
            )
            run = ablator.train_and_evaluate(
                model,
                train_loader,
                test_loader,
                num_epochs=EPOCHS,
                lr=LR,
                optimizer_class=torch.optim.AdamW,
                seed=seed,
                verbose=False,
            )
            frame = run["test_predictions"]
            frame.to_csv(path, index=False)
            del run, model, train_loader, test_loader
            gc.collect()
            if ablator.device.type == "cuda":
                torch.cuda.empty_cache()
            elif ablator.device.type == "mps":
                torch.mps.empty_cache()

        frames[variant] = frame
        y = frame["True Label"].to_numpy(int)
        p = frame["Positive Probability"].to_numpy(float)
        predicted = frame["Predicted Label"].to_numpy(int)
        likelihood = ll(y, p)
        elapsed = (
            previous_minutes.get((seed, variant), float("nan"))
            if cached is not None
            else (time.perf_counter() - started) / 60
        )
        diagnostics.append(
            {
                "seed": seed,
                "variant": variant,
                "geometric_mean_likelihood": likelihood,
                "information_deficit_nats": information_deficit(likelihood),
                "roc_auc": roc_auc_score(y, p),
                "balanced_accuracy": balanced_accuracy_score(y, predicted),
                "parameters": int(variant_table.loc[variant, "parameters"]),
                "minutes": elapsed,
                "cached": cached is not None,
            }
        )
        note = "reused" if cached is not None else f"{elapsed:.2f} min"
        print(
            f"  {variant:<17} a={likelihood:.5f} "
            f"AUC={diagnostics[-1]['roc_auc']:.4f} ({note})"
        )

    # Use the package implementation rather than transcribing the IMV formula.
    matrix = AblationIMV.calculate_imv_matrix(frames)
    seed_matrices.append(matrix)
    for enhanced in VARIANTS:
        for basic in VARIANTS:
            matrix_records.append(
                {
                    "seed": seed,
                    "enhanced": enhanced,
                    "basic": basic,
                    "ablation_imv": matrix.loc[enhanced, basic],
                }
            )


diagnostic_frame = pd.DataFrame(diagnostics)
by_seed = pd.DataFrame(matrix_records)
diagnostic_frame.to_csv(RESULTS / "magic_gamma_variant_diagnostics.csv", index=False)
by_seed.to_csv(RESULTS / "magic_gamma_ablation_imv_by_seed.csv", index=False)
diagnostic_frame.groupby("variant")[[
    "geometric_mean_likelihood",
    "information_deficit_nats",
    "roc_auc",
    "balanced_accuracy",
]].agg(["mean", "std"])


## 4. Directional ablation IMV

Rows are treated as the enhanced model and columns as the basic model.
Directional IMV is not constrained to be positive: a near-zero or negative
full-versus-ablation value is evidence that the omitted group did not add
held-out information under this model and training budget.

The standard deviation describes variation across the ten complete runs; it is
not a confidence interval.


In [ ]:
mean_matrix = AblationIMV.average_imv_matrices(seed_matrices)
matrix_std = pd.DataFrame(
    np.std(
        np.stack([matrix.to_numpy() for matrix in seed_matrices]), axis=0, ddof=1
    ),
    index=mean_matrix.index,
    columns=mean_matrix.columns,
)
mean_matrix.to_csv(RESULTS / "magic_gamma_ablation_imv_directional.csv")
matrix_std.to_csv(RESULTS / "magic_gamma_ablation_imv_directional_std.csv")

print("Mean directional ablation IMV over", len(SEEDS), "seeds")
print("rows = enhanced model; columns = basic model")
mean_matrix


## 5. Figures

The first figure shows all directional pairwise comparisons, using the same
heatmap convention as the MNIST and HAR ablation examples. The second isolates
the full model relative to each feature-group ablation. Error bars are one
standard deviation across seeds.


In [ ]:
fig, ax = plot_ablation_matrix(
    mean_matrix,
    figsize=(8.4, 7.0),
    title=f"MAGIC Gamma feature-group ablation IMV, mean of {len(SEEDS)} seeds",
)
ax.set_xlabel("Basic model")
ax.set_ylabel("Enhanced model")
fig.tight_layout()
matrix_paths = {
    file_format: relative_path(path, start=ARTIFACTS)
    for file_format, path in save_figure(
        fig, FIGURES / "magic_gamma_ablation_imv"
    ).items()
}
plt.show()


full_vs_ablation = (
    by_seed[
        (by_seed["enhanced"] == "FullFeatures")
        & (by_seed["basic"] != "FullFeatures")
    ]
    .groupby("basic", sort=False)["ablation_imv"]
    .agg(["mean", "std"])
    .reindex(VARIANTS[1:])
)
full_vs_ablation.to_csv(RESULTS / "magic_gamma_full_vs_ablation.csv")


fig2, ax2 = plt.subplots(figsize=(7.6, 4.2))
colors = [
    "#2f9cba" if value >= 0 else "#b4464b"
    for value in full_vs_ablation["mean"]
]
ax2.bar(
    full_vs_ablation.index,
    full_vs_ablation["mean"],
    yerr=full_vs_ablation["std"].fillna(0),
    capsize=3,
    color=colors,
)
ax2.axhline(0, color="0.3", linewidth=1)
ax2.set_ylabel("Directional IMV (FullFeatures enhanced)")
ax2.set_title("Full MAGIC Gamma model relative to each feature-group ablation")
ax2.tick_params(axis="x", rotation=18)
fig2.tight_layout()
comparison_paths = {
    file_format: relative_path(path, start=ARTIFACTS)
    for file_format, path in save_figure(
        fig2, FIGURES / "magic_gamma_full_vs_ablation"
    ).items()
}
plt.show()
full_vs_ablation
